# 01 - Detection and segmentation inspector

**Question this notebook answers:** is the pretrained model actually finding the
cow, and does the mask cover the back?

Every filter threshold in `PrepareConfig` is visible and adjustable here. You
tune them against pictures, then commit one batch run.

| Panel | What to check |
|---|---|
| 1. raw frame | Is this a usable lateral pass? |
| 2. detections | One cow found? Right animal? |
| 3. crop | Whole body, no cut-off rump or head? |
| 4. mask overlay | Does the mask follow the **back line**, or bleed into rails/shadow? |

The detector supplies pre-annotation only. Its output is never a posture label.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_colwidth", 60)
print("project root:", PROJECT_ROOT)

## Run configuration

In [ ]:
PREVIEW_ONLY = True
MAX_SAMPLES = 20
SAVE_OUTPUTS = False

SOURCES_CSV = PROJECT_ROOT / "data" / "sources_selected.csv"   # from notebook 00
PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"
MANIFEST_CSV = PROJECT_ROOT / "data" / "manifest.csv"

TARGET_FPS = 1.0
MODEL_NAME = "yolo11n-seg.pt"   # segmentation checkpoint; "none" treats frames as ready crops

In [ ]:
def guard_save(what: str) -> bool:
    """Refuse to write anything while the notebook is in preview mode."""
    if PREVIEW_ONLY or not SAVE_OUTPUTS:
        print(f"PREVIEW MODE - not writing {what}. Set PREVIEW_ONLY=False and SAVE_OUTPUTS=True to commit.")
        return False
    return True

## 1. Load the detector

The `cow` class is resolved **by name**, not by a hard-coded COCO index. A
different checkpoint can reorder its classes, and a silently wrong index would
produce a plausible-looking but meaningless dataset.

In [ ]:
from cowarch.detect import load_detector

detector, cow_class = load_detector(MODEL_NAME)
if detector is None:
    print("detector disabled - every frame is treated as an existing crop")
else:
    print(f"checkpoint: {MODEL_NAME}")
    print(f"resolved 'cow' -> class index {cow_class}")
    print(f"checkpoint reports {len(detector.names)} classes")

## 2. Filter thresholds

These are the only knobs. Change them, re-run the panels below, and watch the
reject reasons move.

In [ ]:
from cowarch.prepare import PrepareConfig

config = PrepareConfig(
    confidence=0.35,       # detection confidence floor
    padding=0.03,          # crop padding as a fraction of the box
    min_area_ratio=0.05,   # cow too small in frame -> reject
    max_area_ratio=0.90,   # cow fills the frame, body likely cut off -> reject
    side_aspect=1.40,      # W/H below this reads as oblique/frontal
    hard_side_filter=False,# False = record the hint, let a human judge
    reject_border=False,   # True = drop frames where the box touches the edge
    dedup_hamming=4,       # perceptual-hash distance for near-duplicate frames
)
pd.Series(config.as_dict())

`hard_side_filter=False` is deliberate. Aspect ratio is a hint, not a view
classifier: a stretched-out walking cow and a cow standing at an angle can land
on the same ratio. Leaving it off records `view_hint` and lets the labeling step
mark genuinely unusable frames `invalid`.

## 3. Inspect single frames

The six-panel view. Panels 5 and 6 (topline, keypoint geometry) belong to the
next notebook and appear empty here unless a mask exists.

In [ ]:
from cowarch.frames import iter_frames, sample_frames
from cowarch.prepare import blank_record, process_frame
from cowarch.viz import inspect_frame

sources = pd.read_csv(SOURCES_CSV, keep_default_na=False, comment="#")
sources["path"] = [
    row.get("resolved_path", "") or row.get("local_path", "") for _, row in sources.iterrows()
]
print(f"{len(sources)} selected sources")
license_columns = [c for c in ["license_status", "license_name", "license"] if c in sources]
sources[["source_id", "kind", *license_columns]]

In [ ]:
def inspect(source_id: str, n: int = 6, fps: float = TARGET_FPS, detection_index=None):
    """Preview frames; set detection_index to inspect one cow in a multi-cow frame."""
    row = sources.loc[sources["source_id"] == source_id].iloc[0]
    path = Path(row["path"])
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    outcomes = []
    last_hash = None
    for frame_idx, frame, name in sample_frames(path, row["kind"], fps, n):
        record = blank_record(source_id, source_id, frame_idx, name)
        outcome = process_frame(
            frame, record, detector=detector, cow_class=cow_class,
            config=config, last_hash=last_hash, detection_index=detection_index,
        )
        if outcome.accepted:
            last_hash = outcome.hash_value
        outcomes.append(outcome)
        inspect_frame(outcome)
        plt.show()
    return outcomes

SOURCE_TO_INSPECT = sources["source_id"].iloc[0] if len(sources) else None
outcomes = inspect(SOURCE_TO_INSPECT, n=4) if SOURCE_TO_INSPECT else []

## 4. Reject reasons across a larger sample

If one reason dominates, the threshold behind it is probably wrong for this
footage. Common cases:

- `multiple_cows` high -> the camera sees the queue as well as the passing cow
- `bbox_area` high -> distance or zoom is off; adjust `min_area_ratio`
- `near_duplicate` high -> `TARGET_FPS` is above the pace of the animal
- `no_cow` high -> confidence too strict, or the footage is not what you think

In [ ]:
def tally(limit_per_source: int = MAX_SAMPLES, fps: float = TARGET_FPS):
    sampled_outcomes = []
    for _, source in sources.iterrows():
        path = Path(source["path"])
        if not path.is_absolute():
            path = PROJECT_ROOT / path
        if not path.exists():
            continue
        last_hash = None
        for frame_idx, frame, name in sample_frames(path, source["kind"], fps, limit_per_source):
            record = blank_record(source["source_id"], source["source_id"], frame_idx, name)
            outcome = process_frame(
                frame, record, detector=detector, cow_class=cow_class,
                config=config, last_hash=last_hash,
            )
            if outcome.accepted:
                last_hash = outcome.hash_value
            sampled_outcomes.append(outcome)
    return sampled_outcomes

sampled_outcomes = tally()
sampled = pd.DataFrame([outcome.record for outcome in sampled_outcomes])
if len(sampled):
    reasons = sampled["reject_reason"].replace("", "ACCEPTED").value_counts()
    display(reasons.to_frame("frames"))
    ax = reasons.plot.barh(figsize=(7, 3.2), color="#2e86ab")
    ax.set_xlabel("frames")
    ax.set_title(f"outcome over {len(sampled)} sampled frames")
    plt.tight_layout(); plt.show()
    print(f"accept rate: {(sampled['reject_reason'] == '').mean():.1%}")

### Accepted crops at a glance

Green border = accepted. Scan for anything that slipped through: two cows, a rear view, a cut-off body.

In [ ]:
from cowarch.viz import thumbnail_grid

accepted_outcomes = [outcome for outcome in sampled_outcomes if outcome.accepted]
rejected_outcomes = [outcome for outcome in sampled_outcomes if not outcome.accepted]
print(f"{len(accepted_outcomes)} accepted in this sample")
if accepted_outcomes:
    fig = thumbnail_grid(
        [outcome.crop for outcome in accepted_outcomes[:15]],
        [outcome.record["sample_id"] for outcome in accepted_outcomes[:15]],
        colors=["#2e7d32"] * min(15, len(accepted_outcomes)),
    )
    fig.suptitle("Accepted crops (memory only)", y=1.01)
    plt.show()

if rejected_outcomes:
    rejects_to_show = []
    for reason in sorted({outcome.reject_reason for outcome in rejected_outcomes}):
        rejects_to_show.extend(
            [outcome for outcome in rejected_outcomes if outcome.reject_reason == reason][:2]
        )
    fig = thumbnail_grid(
        [outcome.crop if outcome.crop is not None else outcome.frame for outcome in rejects_to_show],
        [f"{outcome.reject_reason}: {outcome.record['sample_id']}" for outcome in rejects_to_show],
        colors=["#d1495b"] * len(rejects_to_show),
    )
    fig.suptitle("Rejected examples by reason (memory only)", y=1.01)
    plt.show()

## 5. Commit the batch run

Once the panels look right, run the **same code path** over everything through the
CLI backend. `scripts/02_prepare.py` calls the identical `process_frame`, so what
you inspected is what gets written.

In [ ]:
import subprocess

command = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "02_prepare.py"),
    "--sources", str(SOURCES_CSV),
    "--output-dir", str(PREPARED_DIR),
    "--manifest", str(MANIFEST_CSV),
    "--target-fps", str(TARGET_FPS),
    "--model", MODEL_NAME,
    "--confidence", str(config.confidence),
    "--padding", str(config.padding),
    "--min-area-ratio", str(config.min_area_ratio),
    "--max-area-ratio", str(config.max_area_ratio),
    "--side-aspect", str(config.side_aspect),
    "--dedup-hamming", str(config.dedup_hamming),
]
if config.hard_side_filter:
    command.append("--hard-side-filter")
if config.reject_border:
    command.append("--reject-border")

print(" ".join(command))
if guard_save(f"{MANIFEST_CSV} and crops under {PREPARED_DIR}"):
    result = subprocess.run(command, cwd=PROJECT_ROOT)
    print("exit code:", result.returncode)

In [ ]:
if MANIFEST_CSV.exists():
    manifest = pd.read_csv(MANIFEST_CSV, keep_default_na=False)
    print(f"{len(manifest)} rows, {manifest['accepted'].astype(str).str.lower().isin(['true','1']).sum()} accepted")
    display(manifest.groupby("source_id")["accepted"].apply(
        lambda s: s.astype(str).str.lower().isin(["true", "1"]).sum()
    ).to_frame("accepted crops"))
    print("\nNext: lock the split with scripts/03_split.py, then 02_back_geometry_inspector.ipynb")